# ORB-SLAM3 baseline on real R&S data (Colab, CPU)

Runs `tracks/orbslam3` end to end on one sequence: build (cached on Drive), convert, run
stereo-inertial (default and `IMU.fastInit`) and stereo, score with `eval/`, save to Drive.

Needs `dataset_extracted/<seq>/` in the Drive you mount. CPU runtime is enough.
First run builds ORB-SLAM3 (~15-25 min); later runs restore the cached build.

In [ ]:
SEQ = "seq01"
DRIVE_DATA = f"/content/drive/MyDrive/dataset_extracted/{SEQ}"
DRIVE_CACHE = "/content/drive/MyDrive/rs-cache"
DRIVE_RESULTS = f"/content/drive/MyDrive/rs-results/{SEQ}"
REPO = "/content/rs"

from google.colab import drive
drive.mount("/content/drive")
import os
assert os.path.isdir(DRIVE_DATA), f"{DRIVE_DATA} missing: mount the Drive that holds dataset_extracted/"
print(sorted(os.listdir(DRIVE_DATA)))

In [ ]:
%%bash -s "$REPO"
set -e
[ -d "$1/.git" ] || git clone -q --depth 1 https://github.com/AliAlpOezer/rs-hackatum-2026 "$1"
git -C "$1" log --oneline -1
pip install -q evo scipy pyyaml opencv-python-headless

In [ ]:
%%bash -s "$REPO" "$DRIVE_CACHE"
# Build ORB-SLAM3 + Pangolin, or restore them from the Drive cache (same absolute paths).
set -e
cd "$1"
CACHE="$2/orbslam3_build.tgz"
if [ -f "$CACHE" ] && [ ! -d tracks/orbslam3/ORB_SLAM3 ]; then
  echo "restoring cached build"; tar -xzf "$CACHE" -C tracks/orbslam3
fi
time bash tracks/orbslam3/setup.sh --deps > /content/setup.log 2>&1 || { tail -40 /content/setup.log; exit 1; }
tail -2 /content/setup.log
if [ ! -f "$CACHE" ]; then
  mkdir -p "$2"; tar -czf "$CACHE" -C tracks/orbslam3 ORB_SLAM3 Pangolin; ls -lh "$CACHE"
fi

In [ ]:
%%bash -s "$REPO" "$DRIVE_DATA" "$SEQ"
# Convert: default settings and an IMU.fastInit variant; ground truth in camera frame.
set -e
cd "$1"
mkdir -p /content/raw && cp -r "$2" /content/raw/
python data/tools/zed_to_euroc.py /content/raw/$3 /content/euroc/$3 --gray
python data/tools/zed_to_euroc.py /content/raw/$3 /content/euroc_fastinit/$3 --gray --fast-init
python data/tools/euroc_to_seq.py /content/euroc/$3 data/$3
cat /content/euroc/$3/conversion.yaml

In [ ]:
%%bash -s "$REPO" "$SEQ"
# Run. run.sh names results by the sequence folder, so the fastInit run goes to its own dir.
cd "$1"
export PYTHON=python3
for mode in stereo_inertial stereo; do
  echo "=== $mode"; bash tracks/orbslam3/run.sh /content/euroc/$2 $mode
done
echo "=== stereo_inertial + fastInit"
RESULTS_DIR=/content/results_fastinit bash tracks/orbslam3/run.sh /content/euroc_fastinit/$2 stereo_inertial \
  && cp /content/results_fastinit/$2/orbslam3_stereo_inertial.txt results/$2/orbslam3_stereo_inertial_fastinit.txt
ls results/$2

In [ ]:
%%bash -s "$REPO" "$SEQ"
cd "$1"
python eval/evaluate.py data/$2/gt.txt results/$2/*.txt --metric | tee results/$2/eval.txt

In [ ]:
%%bash -s "$REPO" "$SEQ" "$DRIVE_RESULTS"
# Save trajectories, raw outputs, logs and scores to Drive.
mkdir -p "$3"
cp -r "$1/results/$2/." "$3/"
cp -r /content/results_fastinit/$2/raw "$3/raw_fastinit" 2>/dev/null
cp /content/euroc/$2/conversion.yaml "$3/"
ls -R "$3" | head -30